# 01. Train

Each run is saved on Drive in `MyDrive/vihsd-lora/outputs/runs/<run name>` and logged to W&B.

- **Checkpoints:** LoRA saves every epoch, full fine-tune every 2 epochs. Drive keeps only the latest checkpoint.
- **Best model:** saved to `best/` as soon as the dev macro F1 improves.
- **Session disconnected:** re-run the whole notebook with the same run. Training resumes from the latest checkpoint and continues the same W&B run.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Change this to your repo URL. For a private repo, add a GITHUB_TOKEN secret on Colab.
REPO_URL = "https://github.com/YOUR-USERNAME/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "YOUR-USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repo URL, then run this cell again.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab ships an old torchao that makes peft fail when attaching LoRA. This repo does not use torchao, so remove it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Platform:", detect_platform())
print("GPU:", describe_gpu())
print("Secrets found:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Results are saved to:", settings.output_root)
print(f"Free space: {free_gb} GB")
if free_gb < 6:
    print("Warning: less than 6GB left. A full fine-tune run needs about 3GB, so clean up Drive and empty the trash.")

## Choose runs

| Run | Method | r | Epochs |
|---|---|---|---|
| A | LoRA | 8 | 10 |
| B | Full fine-tune | | 10 |
| C | LoRA | 2 | 10 |
| D | LoRA | 16 | 10 |
| E | LoRA | 8 | 30 |

You can train several runs back to back, or give each team member one run if you share a W&B project.

In [ ]:
RUNS_TO_TRAIN = ["A"]
SEED = 42
WANDB_TEAM = None  # Set your W&B team name if the whole team shares one workspace
SAVE_EVERY_EPOCHS = None  # Leave empty for the defaults: LoRA every epoch, full fine-tune every 2 epochs

settings.wandb_entity = WANDB_TEAM or settings.wandb_entity
settings.save_every_epochs = SAVE_EVERY_EPOCHS

In [ ]:
import gc
import pandas as pd
import torch
from vihsd_lora.train import train_run

summaries = []
for key in RUNS_TO_TRAIN:
    summaries.append(train_run(key, seed=SEED, settings=settings))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

columns = ["slug", "trainable_params", "trainable_pct", "dev_accuracy", "dev_macro_f1", "best_epoch",
           "mean_epoch_seconds", "gpu_peak_mb", "saved_mb"]
display(pd.DataFrame(summaries)[columns])

## View the charts

Open the W&B link printed during training. Per-epoch charts are in the `by_epoch` group: `dev_macro_f1`, `dev_accuracy`, `train_loss`, `epoch_seconds` and `gpu_peak_mb`. Select several runs in the Runs table to compare them on the same chart.